# Spatio-Flux Test Suite — compositional multiscale scenarios

_Investigation `spatio-flux-test-suite` — coder reproduction notebook._

The spatio-flux reference application reproduced as one investigation: 19
studies, each a composition scenario from the original test suite, arranged
standalone processes → pairs → triples → reference demos.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/spatio-flux/spatio-flux').is_dir():
    REPO = Path('/home/runner/work/spatio-flux/spatio-flux')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from spatio_flux.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: `monod_kinetics`

**Question.** Field-only baseline: Monod uptake/growth on a well-mixed substrate pool (no spatial lattice, no particles). Use to sanity-check kinetics + mass balance.

**Purpose.** Runs the spatio_flux.composites.metabolism.monod_kinetics composite (standalone process).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.metabolism.monod_kinetics` | 0 | model_id=overflow_metabolism |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.metabolism.monod_kinetics`**

Monod uptake and growth kinetics on a well-mixed pool of glucose and acetate — the simplest field-only metabolism model, with no spatial lattice or particles.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_metabolism_monod_kinetics = {
    'model_id': 'overflow_metabolism',
    'interval': 0.1,
    'glucose' : 10.0,
    'acetate' : 0.0,
    'biomass' : 0.1,
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: monod_kinetics ===
STUDY = 'monod_kinetics'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_metabolism_monod_kinetics if 'ov_spatio_flux_composites_metabolism_monod_kinetics' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_metabolism_monod_kinetics = materialize_composite('spatio_flux.composites.metabolism.monod_kinetics', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_metabolism_monod_kinetics['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.metabolism.monod_kinetics) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**monod_kinetics.svg**


In [ ]:
# monod_kinetics.svg
show_viz(_render_one('image:visualizations/monod_kinetics.svg', {}, RUNS_DB, STUDY_YAML))

**monod_kinetics.png**


In [ ]:
# monod_kinetics.png
show_viz(_render_one('image:charts/monod_kinetics.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| MONOD_KINETICS-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['monod_kinetics.png', 'monod_kinetics_viz.png', 'monod_kinetics_schema.json', 'monod_kinetics_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `ecoli_core_dfba`

**Question.** Single-cell metabolism baseline: dynamic FBA for E. coli core with external glucose/acetate and biomass over time (no space, no particles).

**Purpose.** Runs the spatio_flux.composites.metabolism.ecoli_core_dfba composite (standalone process).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.metabolism.ecoli_core_dfba` | 0 | model_id=ecoli core, glucose=10.0, acetate=0.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.metabolism.ecoli_core_dfba`**

Dynamic FBA of a single E. coli core cell in a well-mixed medium, tracking extracellular glucose and acetate as biomass grows — the simplest single-cell metabolism model, with no space or particles.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_metabolism_ecoli_core_dfba = {
    'model_id': 'ecoli core',
    'glucose' : 10.0,
    'acetate' : 0.0,
    'biomass' : 0.1,
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: ecoli_core_dfba ===
STUDY = 'ecoli_core_dfba'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_metabolism_ecoli_core_dfba if 'ov_spatio_flux_composites_metabolism_ecoli_core_dfba' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_metabolism_ecoli_core_dfba = materialize_composite('spatio_flux.composites.metabolism.ecoli_core_dfba', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_metabolism_ecoli_core_dfba['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.metabolism.ecoli_core_dfba) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**ecoli_core_dfba.svg**


In [ ]:
# ecoli_core_dfba.svg
show_viz(_render_one('image:visualizations/ecoli_core_dfba.svg', {}, RUNS_DB, STUDY_YAML))

**ecoli_core_dfba.png**


In [ ]:
# ecoli_core_dfba.png
show_viz(_render_one('image:charts/ecoli_core_dfba.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| ECOLI_CORE_DFBA-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['ecoli_core_dfba.png', 'ecoli_core_dfba_viz.png', 'ecoli_core_dfba_schema.json', 'ecoli_core_dfba_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `ecoli_dfba`

**Question.** Single-cell metabolism (large model): dynamic FBA using iAF1260 with tracked extracellular fields (glucose/formate) and biomass. Stress-tests solver + exchange wiring.

**Purpose.** Runs the spatio_flux.composites.metabolism.ecoli_dfba composite (standalone process).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.metabolism.ecoli_dfba` | 0 | model_id=ecoli, glucose=10.0, formate=5.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.metabolism.ecoli_dfba`**

Dynamic FBA of a single E. coli cell using the genome-scale iAF1260 model, tracking extracellular glucose and formate as biomass grows in a well-mixed medium.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_metabolism_ecoli_dfba = {
    'model_id': 'ecoli',
    'glucose' : 10.0,
    'formate' : 5.0,
    'biomass' : 0.1,
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: ecoli_dfba ===
STUDY = 'ecoli_dfba'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_metabolism_ecoli_dfba if 'ov_spatio_flux_composites_metabolism_ecoli_dfba' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_metabolism_ecoli_dfba = materialize_composite('spatio_flux.composites.metabolism.ecoli_dfba', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_metabolism_ecoli_dfba['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.metabolism.ecoli_dfba) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**ecoli_dfba.svg**


In [ ]:
# ecoli_dfba.svg
show_viz(_render_one('image:visualizations/ecoli_dfba.svg', {}, RUNS_DB, STUDY_YAML))

**ecoli_dfba.png**


In [ ]:
# ecoli_dfba.png
show_viz(_render_one('image:charts/ecoli_dfba.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| ECOLI_DFBA-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['ecoli_dfba.png', 'ecoli_dfba_viz.png', 'ecoli_dfba_schema.json', 'ecoli_dfba_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `yeast_dfba`

**Question.** Single-cell metabolism (yeast): dynamic FBA using iMM904 with extracellular glucose and biomass. Cross-model check of the dFBA pipeline.

**Purpose.** Runs the spatio_flux.composites.metabolism.yeast_dfba composite (standalone process).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.metabolism.yeast_dfba` | 0 | model_id=yeast, glucose=5.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.metabolism.yeast_dfba`**

Dynamic FBA of a single yeast cell using the genome-scale iMM904 model, tracking extracellular glucose as biomass grows in a well-mixed medium.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_metabolism_yeast_dfba = {
    'model_id': 'yeast',
    'glucose' : 5.0,
    'biomass' : 0.1,
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: yeast_dfba ===
STUDY = 'yeast_dfba'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_metabolism_yeast_dfba if 'ov_spatio_flux_composites_metabolism_yeast_dfba' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_metabolism_yeast_dfba = materialize_composite('spatio_flux.composites.metabolism.yeast_dfba', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_metabolism_yeast_dfba['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.metabolism.yeast_dfba) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**yeast_dfba.svg**


In [ ]:
# yeast_dfba.svg
show_viz(_render_one('image:visualizations/yeast_dfba.svg', {}, RUNS_DB, STUDY_YAML))

**yeast_dfba.png**


In [ ]:
# yeast_dfba.png
show_viz(_render_one('image:charts/yeast_dfba.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| YEAST_DFBA-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['yeast_dfba.png', 'yeast_dfba_viz.png', 'yeast_dfba_schema.json', 'yeast_dfba_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `diffusion_process`

**Question.** Field transport primitive: finite-volume diffusion/advection on a 2D lattice. Use to validate boundary conditions, stability, and transport timescales.

**Purpose.** Runs the spatio_flux.composites.spatial.diffusion_process composite (standalone process).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.spatial.diffusion_process` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.spatial.diffusion_process`**

Finite-volume diffusion and advection of solute fields on a 2D lattice — glucose diffuses while a seeded band of dissolved biomass diffuses and advects — the field-transport building block underlying the spatial composites.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_spatial_diffusion_process = {
    'n_bins'        : [10, 20],
    'bounds'        : [40.0, 80.0],
    'diffusion_rate': 0.05,
    'advection_rate': [0.0, 0.2],
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: diffusion_process ===
STUDY = 'diffusion_process'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_spatial_diffusion_process if 'ov_spatio_flux_composites_spatial_diffusion_process' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_spatial_diffusion_process = materialize_composite('spatio_flux.composites.spatial.diffusion_process', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_spatial_diffusion_process['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.spatial.diffusion_process) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**diffusion_process.svg**


In [ ]:
# diffusion_process.svg
show_viz(_render_one('image:visualizations/diffusion_process.svg', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| DIFFUSION_PROCESS-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['diffusion_process_video.gif', 'diffusion_process_viz.png', 'diffusion_process_schema.json', 'diffusion_process_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `brownian_particles`

**Question.** Particle-only baseline: Brownian motion of agents with mass in continuous space (no fields, no metabolism). Checks integrator + particle state schema.

**Purpose.** Runs the spatio_flux.composites.particles.brownian_particles composite (standalone process).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.particles.brownian_particles` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.particles.brownian_particles`**

Brownian motion of mass-carrying agents in continuous space — the particle-only building block, with no fields or metabolism.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_particles_brownian_particles = {
    'n_bins'        : [10, 10],
    'bounds'        : [50.0, 50.0],
    'n_particles'   : 1,
    'time_interval' : 0.1,
    'diffusion_rate': 0.5,
    'add_rate'      : 0.01,
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: brownian_particles ===
STUDY = 'brownian_particles'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_particles_brownian_particles if 'ov_spatio_flux_composites_particles_brownian_particles' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_particles_brownian_particles = materialize_composite('spatio_flux.composites.particles.brownian_particles', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_particles_brownian_particles['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.particles.brownian_particles) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**brownian_particles.svg**


In [ ]:
# brownian_particles.svg
show_viz(_render_one('image:visualizations/brownian_particles.svg', {}, RUNS_DB, STUDY_YAML))

**brownian_particles_snapshots.png**


In [ ]:
# brownian_particles_snapshots.png
show_viz(_render_one('image:charts/brownian_particles_snapshots.png', {}, RUNS_DB, STUDY_YAML))

**brownian_particles_particles_traces.png**


In [ ]:
# brownian_particles_particles_traces.png
show_viz(_render_one('image:charts/brownian_particles_particles_traces.png', {}, RUNS_DB, STUDY_YAML))

**brownian_particles_mass.png**


In [ ]:
# brownian_particles_mass.png
show_viz(_render_one('image:charts/brownian_particles_mass.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| BROWNIAN_PARTICLES-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['brownian_particles_video.gif', 'brownian_particles_mass.png', 'brownian_particles_snapshots.png', 'brownian_particles_particles_traces.png', 'brownian_particles_viz.png', 'brownian_particles_schema.json', 'brownian_particles_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `newtonian_particles`

**Question.** Physics-only baseline (Pymunk): rigid-body particles with collisions/crowding in continuous space. Use to validate contact dynamics + boundary enforcement.

**Purpose.** Runs the spatio_flux.composites.particles.newtonian_particles composite (standalone process).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.particles.newtonian_particles` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.particles.newtonian_particles`**

Rigid-body particles under gravity that collide and crowd in continuous space via a Pymunk physics engine — the mechanics-only building block, with no fields or metabolism.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_particles_newtonian_particles = {
    'n_particles': 1,
    'gravity'    : -0.2,
    'elasticity' : 0.1,
    'add_rate'   : 0.02,
    'bounds'     : [50.0, 50.0],
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: newtonian_particles ===
STUDY = 'newtonian_particles'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_particles_newtonian_particles if 'ov_spatio_flux_composites_particles_newtonian_particles' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_particles_newtonian_particles = materialize_composite('spatio_flux.composites.particles.newtonian_particles', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_particles_newtonian_particles['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.particles.newtonian_particles) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**newtonian_particles.svg**


In [ ]:
# newtonian_particles.svg
show_viz(_render_one('image:visualizations/newtonian_particles.svg', {}, RUNS_DB, STUDY_YAML))

**newtonian_particles_particles_traces.png**


In [ ]:
# newtonian_particles_particles_traces.png
show_viz(_render_one('image:charts/newtonian_particles_particles_traces.png', {}, RUNS_DB, STUDY_YAML))

**newtonian_particles_mass.png**


In [ ]:
# newtonian_particles_mass.png
show_viz(_render_one('image:charts/newtonian_particles_mass.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| NEWTONIAN_PARTICLES-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['newtonian_particles_mass.png', 'newtonian_particles_video.gif', 'newtonian_particles_particles_traces.png', 'newtonian_particles_viz.png', 'newtonian_particles_schema.json', 'newtonian_particles_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `community_dfba`

**Question.** Multi-agent well-mixed community: several independent dFBA instances share the same extracellular pools, creating competition / cross-feeding dynamics without space.

**Purpose.** Runs the spatio_flux.composites.metabolism.community_dfba composite (pair composition).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.metabolism.community_dfba` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.metabolism.community_dfba`**

A well-mixed microbial community where several dFBA species and a Monod-kinetic population share the same extracellular substrate pools, producing competition and cross-feeding without spatial structure.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_metabolism_community_dfba = {
    'dt'              : 1.0,
    'kinetic_model_id': 'acetate_only',
    'initial_biomass' : 0.1,
    'glucose'         : 10.0,
    'acetate'         : 0.0,
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: community_dfba ===
STUDY = 'community_dfba'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_metabolism_community_dfba if 'ov_spatio_flux_composites_metabolism_community_dfba' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_metabolism_community_dfba = materialize_composite('spatio_flux.composites.metabolism.community_dfba', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_metabolism_community_dfba['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.metabolism.community_dfba) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**community_dfba.svg**


In [ ]:
# community_dfba.svg
show_viz(_render_one('image:visualizations/community_dfba.svg', {}, RUNS_DB, STUDY_YAML))

**community_dfba.png**


In [ ]:
# community_dfba.png
show_viz(_render_one('image:charts/community_dfba.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| COMMUNITY_DFBA-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['community_dfba.png', 'community_dfba_viz.png', 'community_dfba_schema.json', 'community_dfba_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `dfba_kinetics_community`

**Question.** Hybrid community (well-mixed): mixes Monod-kinetic agents with dFBA agents in a shared environment. Demonstrates heterogeneous process composition under one schema.

**Purpose.** Runs the spatio_flux.composites.metabolism.dfba_kinetics_community composite (pair composition).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.metabolism.dfba_kinetics_community` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.metabolism.dfba_kinetics_community`**

A well-mixed community pairing a dFBA population with a Monod-kinetic population that share the same extracellular glucose and acetate pools, coupling two metabolic modeling styles in one environment.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_metabolism_dfba_kinetics_community = {
    'dfba_model_id'     : 'ecoli core',
    'kinetic_model_id'  : 'acetate_only',
    'dfba_biomass_id'   : 'dfba biomass',
    'kinetic_biomass_id': 'kinetic biomass',
    'glucose'           : 10.0,
    'acetate'           : 0.0,
    'initial_biomass'   : 0.01,
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: dfba_kinetics_community ===
STUDY = 'dfba_kinetics_community'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_metabolism_dfba_kinetics_community if 'ov_spatio_flux_composites_metabolism_dfba_kinetics_community' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_metabolism_dfba_kinetics_community = materialize_composite('spatio_flux.composites.metabolism.dfba_kinetics_community', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_metabolism_dfba_kinetics_community['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.metabolism.dfba_kinetics_community) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**dfba_kinetics_community.svg**


In [ ]:
# dfba_kinetics_community.svg
show_viz(_render_one('image:visualizations/dfba_kinetics_community.svg', {}, RUNS_DB, STUDY_YAML))

**dfba_kinetics_community.png**


In [ ]:
# dfba_kinetics_community.png
show_viz(_render_one('image:charts/dfba_kinetics_community.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| DFBA_KINETICS_COMMUNITY-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['dfba_kinetics_community.png', 'dfba_kinetics_community_viz.png', 'dfba_kinetics_community_schema.json', 'dfba_kinetics_community_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `spatial_many_dfba`

**Question.** Spatial microenvironment (sitewise dFBA): a lattice where each site runs its own dFBA instance. Useful for validating lattice indexing + per-site state isolation.

**Purpose.** Runs the spatio_flux.composites.spatial.spatial_many_dfba composite (pair composition).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.spatial.spatial_many_dfba` | 0 | model_id=ecoli core |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.spatial.spatial_many_dfba`**

A spatial microenvironment where every lattice site runs its own dFBA instance over local glucose, acetate, and dissolved biomass, resolving metabolism site by site across the grid.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_spatial_spatial_many_dfba = {
    'model_id': 'ecoli core',
    'n_bins'  : [2, 4],
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: spatial_many_dfba ===
STUDY = 'spatial_many_dfba'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_spatial_spatial_many_dfba if 'ov_spatio_flux_composites_spatial_spatial_many_dfba' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_spatial_spatial_many_dfba = materialize_composite('spatio_flux.composites.spatial.spatial_many_dfba', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_spatial_spatial_many_dfba['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.spatial.spatial_many_dfba) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**spatial_many_dfba.svg**


In [ ]:
# spatial_many_dfba.svg
show_viz(_render_one('image:visualizations/spatial_many_dfba.svg', {}, RUNS_DB, STUDY_YAML))

**spatial_many_dfba_timeseries.png**


In [ ]:
# spatial_many_dfba_timeseries.png
show_viz(_render_one('image:charts/spatial_many_dfba_timeseries.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| SPATIAL_MANY_DFBA-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['spatial_many_dfba_timeseries.png', 'spatial_many_dfba_video.gif', 'spatial_many_dfba_viz.png', 'spatial_many_dfba_schema.json', 'spatial_many_dfba_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `spatial_dfba_process`

**Question.** Spatial microenvironment (vectorized dFBA): one spatial dFBA process updates all lattice sites as a single structured state. Demonstrates batched execution + field coupling.

**Purpose.** Runs the spatio_flux.composites.spatial.spatial_dfba_process composite (pair composition).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.spatial.spatial_dfba_process` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.spatial.spatial_dfba_process`**

A spatial microenvironment where a single vectorized dFBA process updates every lattice site at once, with six microbial species arranged in rows across a horizontal glucose gradient.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_spatial_spatial_dfba_process = {
    'n_bins': [5, 6],
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: spatial_dfba_process ===
STUDY = 'spatial_dfba_process'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_spatial_spatial_dfba_process if 'ov_spatio_flux_composites_spatial_spatial_dfba_process' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_spatial_spatial_dfba_process = materialize_composite('spatio_flux.composites.spatial.spatial_dfba_process', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_spatial_spatial_dfba_process['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.spatial.spatial_dfba_process) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**spatial_dfba_process.svg**


In [ ]:
# spatial_dfba_process.svg
show_viz(_render_one('image:visualizations/spatial_dfba_process.svg', {}, RUNS_DB, STUDY_YAML))

**spatial_dfba_process_timeseries.png**


In [ ]:
# spatial_dfba_process_timeseries.png
show_viz(_render_one('image:charts/spatial_dfba_process_timeseries.png', {}, RUNS_DB, STUDY_YAML))

**spatial_dfba_process_model_grid.png**


In [ ]:
# spatial_dfba_process_model_grid.png
show_viz(_render_one('image:charts/spatial_dfba_process_model_grid.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| SPATIAL_DFBA_PROCESS-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['spatial_dfba_process_timeseries.png', 'spatial_dfba_process_model_grid.png', 'spatial_dfba_process_video.gif', 'spatial_dfba_process_viz.png', 'spatial_dfba_process_schema.json', 'spatial_dfba_process_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `comets_diffusion`

**Question.** Two-phase coupling (no particles): spatial dFBA on a lattice with advection–diffusion of substrates + biomass. The classic COMETS field-only scenario.

**Purpose.** Runs the spatio_flux.composites.comets.comets_diffusion composite (pair composition).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.comets.comets_diffusion` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.comets.comets_diffusion`**

The classic COMETS field-only scenario: per-site dFBA on a lattice coupled to advection–diffusion of glucose, acetate, and dissolved biomass, with biomass seeded at the top edge.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_comets_comets_diffusion = {
    'dissolved_model_id': 'ecoli core',
    'n_bins'            : [10, 10],
    'bounds'            : [50.0, 50.0],
    'advection_rate'    : [0.0, 0.2],
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: comets_diffusion ===
STUDY = 'comets_diffusion'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_comets_comets_diffusion if 'ov_spatio_flux_composites_comets_comets_diffusion' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_comets_comets_diffusion = materialize_composite('spatio_flux.composites.comets.comets_diffusion', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_comets_comets_diffusion['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.comets.comets_diffusion) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**comets_diffusion.svg**


In [ ]:
# comets_diffusion.svg
show_viz(_render_one('image:visualizations/comets_diffusion.svg', {}, RUNS_DB, STUDY_YAML))

**comets_diffusion_timeseries.png**


In [ ]:
# comets_diffusion_timeseries.png
show_viz(_render_one('image:charts/comets_diffusion_timeseries.png', {}, RUNS_DB, STUDY_YAML))

**comets_diffusion_snapshots.png**


In [ ]:
# comets_diffusion_snapshots.png
show_viz(_render_one('image:charts/comets_diffusion_snapshots.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| COMETS_DIFFUSION-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['comets_diffusion_timeseries.png', 'comets_diffusion_snapshots.png', 'comets_diffusion_video.gif', 'comets_diffusion_viz.png', 'comets_diffusion_schema.json', 'comets_diffusion_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `br_particles_kinetics`

**Question.** Particle–field coupling (kinetics): Brownian agents sample local lattice concentrations and apply Monod-style exchange, modifying both particle mass and fields.

**Purpose.** Runs the spatio_flux.composites.particles.br_particles_kinetics composite (pair composition).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.particles.br_particles_kinetics` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.particles.br_particles_kinetics`**

Brownian agents diffuse through a lattice of glucose and acetate, taking up substrate by Monod kinetics so that exchange updates both particle mass and the local fields, with agents dividing once they cross a mass threshold.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_particles_br_particles_kinetics = {
    'model_id'               : 'overflow_metabolism',
    'division_mass_threshold': 5.0,
    'n_bins'                 : [10, 10],
    'bounds'                 : [50.0, 50.0],
    'n_particles'            : 1,
    'diffusion_rate'         : 0.25,
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: br_particles_kinetics ===
STUDY = 'br_particles_kinetics'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_particles_br_particles_kinetics if 'ov_spatio_flux_composites_particles_br_particles_kinetics' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_particles_br_particles_kinetics = materialize_composite('spatio_flux.composites.particles.br_particles_kinetics', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_particles_br_particles_kinetics['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.particles.br_particles_kinetics) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**br_particles_kinetics.svg**


In [ ]:
# br_particles_kinetics.svg
show_viz(_render_one('image:visualizations/br_particles_kinetics.svg', {}, RUNS_DB, STUDY_YAML))

**br_particles_kinetics_snapshots.png**


In [ ]:
# br_particles_kinetics_snapshots.png
show_viz(_render_one('image:charts/br_particles_kinetics_snapshots.png', {}, RUNS_DB, STUDY_YAML))

**br_particles_kinetics_particles_traces.png**


In [ ]:
# br_particles_kinetics_particles_traces.png
show_viz(_render_one('image:charts/br_particles_kinetics_particles_traces.png', {}, RUNS_DB, STUDY_YAML))

**br_particles_kinetics_mass.png**


In [ ]:
# br_particles_kinetics_mass.png
show_viz(_render_one('image:charts/br_particles_kinetics_mass.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| BR_PARTICLES_KINETICS-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['br_particles_kinetics_video.gif', 'br_particles_kinetics_mass.png', 'br_particles_kinetics_snapshots.png', 'br_particles_kinetics_particles_traces.png', 'br_particles_kinetics_viz.png', 'br_particles_kinetics_schema.json', 'br_particles_kinetics_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `br_particles_dfba`

**Question.** Particle-embedded metabolism: Brownian agents carry internal dFBA; uptake/secretion couples to fields and biomass accumulates into particle mass/size.

**Purpose.** Runs the spatio_flux.composites.particles.br_particles_dfba composite (pair composition).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.particles.br_particles_dfba` | 0 | particle_model_id=ecoli core |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.particles.br_particles_dfba`**

Brownian agents each run an internal dFBA model, exchanging glucose and acetate with the surrounding fields so that metabolic biomass accumulates as particle mass and drives division, over a vertical glucose gradient.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_particles_br_particles_dfba = {
    'particle_model_id'      : 'ecoli core',
    'division_mass_threshold': 5.0,
    'add_rate'               : 0.1,
    'n_bins'                 : [10, 10],
    'bounds'                 : [50.0, 50.0],
    'n_particles'            : 1,
    'diffusion_rate'         : 0.5,
    'advection_rate'         : [0.0, 0.2],
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: br_particles_dfba ===
STUDY = 'br_particles_dfba'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_particles_br_particles_dfba if 'ov_spatio_flux_composites_particles_br_particles_dfba' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_particles_br_particles_dfba = materialize_composite('spatio_flux.composites.particles.br_particles_dfba', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_particles_br_particles_dfba['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.particles.br_particles_dfba) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**br_particles_dfba.svg**


In [ ]:
# br_particles_dfba.svg
show_viz(_render_one('image:visualizations/br_particles_dfba.svg', {}, RUNS_DB, STUDY_YAML))

**br_particles_dfba_snapshots.png**


In [ ]:
# br_particles_dfba_snapshots.png
show_viz(_render_one('image:charts/br_particles_dfba_snapshots.png', {}, RUNS_DB, STUDY_YAML))

**br_particles_dfba_particles_traces.png**


In [ ]:
# br_particles_dfba_particles_traces.png
show_viz(_render_one('image:charts/br_particles_dfba_particles_traces.png', {}, RUNS_DB, STUDY_YAML))

**br_particles_dfba_mass.png**


In [ ]:
# br_particles_dfba_mass.png
show_viz(_render_one('image:charts/br_particles_dfba_mass.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| BR_PARTICLES_DFBA-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['br_particles_dfba_video.gif', 'br_particles_dfba_mass.png', 'br_particles_dfba_snapshots.png', 'br_particles_dfba_particles_traces.png', 'br_particles_dfba_viz.png', 'br_particles_dfba_schema.json', 'br_particles_dfba_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `comets_br_particles_kinetics`

**Question.** Three-way coupling: spatial dFBA fields + Brownian particle agents that run Monod kinetics locally + diffusion. End-to-end smoke test of the COMETS-style composite stack with kinetic agents.

**Purpose.** Runs the spatio_flux.composites.comets.comets_br_particles_kinetics composite (triple composition).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.comets.comets_br_particles_kinetics` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.comets.comets_br_particles_kinetics`**

A COMETS-style scenario coupling lattice dFBA fields, advection–diffusion, and Brownian agents that run Monod kinetics against local concentrations, exchanging substrate with the fields and dividing at a mass threshold.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_comets_comets_br_particles_kinetics = {
    'division_mass_threshold': 5.0,
    'dissolved_model_id'     : 'ecoli core',
    'kinetic_model_id'       : 'overflow_metabolism',
    'n_particles'            : 1,
    'add_rate'               : 0.1,
    'n_bins'                 : [10, 10],
    'bounds'                 : [50.0, 50.0],
    'diffusion_rate'         : 0.5,
    'advection_rate'         : [0.0, 0.2],
    'initial_min_max'        : {'glucose': (10, 10), 'acetate': (0, 0), 'dissolved biomass': (0, 0.1)},
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: comets_br_particles_kinetics ===
STUDY = 'comets_br_particles_kinetics'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_comets_comets_br_particles_kinetics if 'ov_spatio_flux_composites_comets_comets_br_particles_kinetics' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_comets_comets_br_particles_kinetics = materialize_composite('spatio_flux.composites.comets.comets_br_particles_kinetics', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_comets_comets_br_particles_kinetics['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.comets.comets_br_particles_kinetics) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**comets_br_particles_kinetics.svg**


In [ ]:
# comets_br_particles_kinetics.svg
show_viz(_render_one('image:visualizations/comets_br_particles_kinetics.svg', {}, RUNS_DB, STUDY_YAML))

**comets_br_particles_kinetics_timeseries.png**


In [ ]:
# comets_br_particles_kinetics_timeseries.png
show_viz(_render_one('image:charts/comets_br_particles_kinetics_timeseries.png', {}, RUNS_DB, STUDY_YAML))

**comets_br_particles_kinetics_snapshots.png**


In [ ]:
# comets_br_particles_kinetics_snapshots.png
show_viz(_render_one('image:charts/comets_br_particles_kinetics_snapshots.png', {}, RUNS_DB, STUDY_YAML))

**comets_br_particles_kinetics_mass.png**


In [ ]:
# comets_br_particles_kinetics_mass.png
show_viz(_render_one('image:charts/comets_br_particles_kinetics_mass.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| COMETS_BR_PARTICLES_KINETICS-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['comets_br_particles_kinetics_timeseries.png', 'comets_br_particles_kinetics_mass.png', 'comets_br_particles_kinetics_snapshots.png', 'comets_br_particles_kinetics_video.gif', 'comets_br_particles_kinetics_viz.png', 'comets_br_particles_kinetics_schema.json', 'comets_br_particles_kinetics_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `comets_br_particles_dfba`

**Question.** Full COMETS-style spatial composite: lattice dFBA + advection–diffusion + Brownian particles with internal dFBA + division. End-to-end coupled scenario.

**Purpose.** Runs the spatio_flux.composites.comets.comets_br_particles_dfba composite (triple composition).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.comets.comets_br_particles_dfba` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.comets.comets_br_particles_dfba`**

The full COMETS-style composite: per-site lattice dFBA and advection–diffusion of substrates coupled to Brownian particles that each run an internal dFBA model, exchange with the fields, and divide as biomass accumulates.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_comets_comets_br_particles_dfba = {
    'particle_model_id'      : 'ecoli core',
    'dissolved_model_id'     : 'ecoli core',
    'division_mass_threshold': 3,
    'n_particles'            : 1,
    'add_rate'               : 0.3,
    'n_bins'                 : [10, 10],
    'bounds'                 : [50.0, 50.0],
    'diffusion_rate'         : 0.5,
    'field_advection_rate'   : [0.0, 0.2],
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: comets_br_particles_dfba ===
STUDY = 'comets_br_particles_dfba'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_comets_comets_br_particles_dfba if 'ov_spatio_flux_composites_comets_comets_br_particles_dfba' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_comets_comets_br_particles_dfba = materialize_composite('spatio_flux.composites.comets.comets_br_particles_dfba', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_comets_comets_br_particles_dfba['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.comets.comets_br_particles_dfba) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**comets_br_particles_dfba.svg**


In [ ]:
# comets_br_particles_dfba.svg
show_viz(_render_one('image:visualizations/comets_br_particles_dfba.svg', {}, RUNS_DB, STUDY_YAML))

**comets_br_particles_dfba_timeseries.png**


In [ ]:
# comets_br_particles_dfba_timeseries.png
show_viz(_render_one('image:charts/comets_br_particles_dfba_timeseries.png', {}, RUNS_DB, STUDY_YAML))

**comets_br_particles_dfba_snapshots.png**


In [ ]:
# comets_br_particles_dfba_snapshots.png
show_viz(_render_one('image:charts/comets_br_particles_dfba_snapshots.png', {}, RUNS_DB, STUDY_YAML))

**comets_br_particles_dfba_mass.png**


In [ ]:
# comets_br_particles_dfba_mass.png
show_viz(_render_one('image:charts/comets_br_particles_dfba_mass.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| COMETS_BR_PARTICLES_DFBA-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['comets_br_particles_dfba_timeseries.png', 'comets_br_particles_dfba_mass.png', 'comets_br_particles_dfba_snapshots.png', 'comets_br_particles_dfba_video.gif', 'comets_br_particles_dfba_viz.png', 'comets_br_particles_dfba_schema.json', 'comets_br_particles_dfba_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `comets_nt_particles_dfba`

**Question.** Mechanochemical + metabolic coupling: Pymunk particles move/collide while COMETS fields diffuse; particles run metabolism (via exchange) against local concentrations.

**Purpose.** Runs the spatio_flux.composites.comets.comets_nt_particles_dfba composite (triple composition).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.comets.comets_nt_particles_dfba` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.comets.comets_nt_particles_dfba`**

Pymunk rigid-body particles that collide and settle under gravity while COMETS fields diffuse and a lattice kinetics process consumes substrate; each particle runs an internal dFBA model, exchanging with the local fields and dividing as it grows.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_comets_comets_nt_particles_dfba = {
    'particle_model_id' : 'ecoli core',
    'dissolved_model_id': 'ecoli core',
    'n_particles'       : 2,
    'bounds'            : [50.0, 50.0],
    'n_bins'            : [20, 20],
    'advection_rate'    : [0.0, 0.2],
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: comets_nt_particles_dfba ===
STUDY = 'comets_nt_particles_dfba'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_comets_comets_nt_particles_dfba if 'ov_spatio_flux_composites_comets_comets_nt_particles_dfba' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_comets_comets_nt_particles_dfba = materialize_composite('spatio_flux.composites.comets.comets_nt_particles_dfba', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_comets_comets_nt_particles_dfba['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.comets.comets_nt_particles_dfba) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**comets_nt_particles_dfba.svg**


In [ ]:
# comets_nt_particles_dfba.svg
show_viz(_render_one('image:visualizations/comets_nt_particles_dfba.svg', {}, RUNS_DB, STUDY_YAML))

**comets_nt_particles_dfba_timeseries.png**


In [ ]:
# comets_nt_particles_dfba_timeseries.png
show_viz(_render_one('image:charts/comets_nt_particles_dfba_timeseries.png', {}, RUNS_DB, STUDY_YAML))

**comets_nt_particles_dfba_snapshots.png**


In [ ]:
# comets_nt_particles_dfba_snapshots.png
show_viz(_render_one('image:charts/comets_nt_particles_dfba_snapshots.png', {}, RUNS_DB, STUDY_YAML))

**comets_nt_particles_dfba_mass.png**


In [ ]:
# comets_nt_particles_dfba_mass.png
show_viz(_render_one('image:charts/comets_nt_particles_dfba_mass.png', {}, RUNS_DB, STUDY_YAML))

**comets_nt_particles_dfba_mass_submasses.png**


In [ ]:
# comets_nt_particles_dfba_mass_submasses.png
show_viz(_render_one('image:charts/comets_nt_particles_dfba_mass_submasses.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| COMETS_NT_PARTICLES_DFBA-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['comets_nt_particles_dfba_timeseries.png', 'comets_nt_particles_dfba_mass.png', 'comets_nt_particles_dfba_mass_submasses.png', 'comets_nt_particles_dfba_video.gif', 'comets_nt_particles_dfba_snapshots.png', 'comets_nt_particles_dfba_viz.png', 'comets_nt_particles_dfba_schema.json', 'comets_nt_particles_dfba_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `spatioflux_reference_demo`

**Question.** SpatioFlux demonstration reference composite: Newtonian motile particles + particle–field exchange + internal multi-dFBA + Monod/diffusion fields + mass-aggregated division.

**Purpose.** Runs the spatio_flux.composites.reference.spatioflux_reference_demo composite (reference demo).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.reference.spatioflux_reference_demo` | 0 | n_bins=[10, 10] |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.reference.spatioflux_reference_demo`**

The flagship SpatioFlux reference model: Newtonian particles, each carrying a two-strain E. coli community with its own internal dFBA, swim through diffusing glucose and acetate fields, exchange substrate locally, and divide by splitting sub-masses between daughters.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_reference_spatioflux_reference_demo = {
    'bounds'     : [50.0, 50.0],
    'n_bins'     : [10, 10],
    'depth'      : 0.04,
    'n_particles': 1,
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: spatioflux_reference_demo ===
STUDY = 'spatioflux_reference_demo'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_reference_spatioflux_reference_demo if 'ov_spatio_flux_composites_reference_spatioflux_reference_demo' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_reference_spatioflux_reference_demo = materialize_composite('spatio_flux.composites.reference.spatioflux_reference_demo', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_reference_spatioflux_reference_demo['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.reference.spatioflux_reference_demo) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**spatioflux_reference_demo.svg**


In [ ]:
# spatioflux_reference_demo.svg
show_viz(_render_one('image:visualizations/spatioflux_reference_demo.svg', {}, RUNS_DB, STUDY_YAML))

**spatioflux_reference_demo_timeseries.png**


In [ ]:
# spatioflux_reference_demo_timeseries.png
show_viz(_render_one('image:charts/spatioflux_reference_demo_timeseries.png', {}, RUNS_DB, STUDY_YAML))

**spatioflux_reference_demo_snapshots.png**


In [ ]:
# spatioflux_reference_demo_snapshots.png
show_viz(_render_one('image:charts/spatioflux_reference_demo_snapshots.png', {}, RUNS_DB, STUDY_YAML))

**spatioflux_reference_demo_mass.png**


In [ ]:
# spatioflux_reference_demo_mass.png
show_viz(_render_one('image:charts/spatioflux_reference_demo_mass.png', {}, RUNS_DB, STUDY_YAML))

**spatioflux_reference_demo_mass_submasses.png**


In [ ]:
# spatioflux_reference_demo_mass_submasses.png
show_viz(_render_one('image:charts/spatioflux_reference_demo_mass_submasses.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| SPATIOFLUX_REFERENCE_DEMO-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['spatioflux_reference_demo_timeseries.png', 'spatioflux_reference_demo_mass.png', 'spatioflux_reference_demo_mass_submasses.png', 'spatioflux_reference_demo_video.gif', 'spatioflux_reference_demo_snapshots.png', 'spatioflux_reference_demo_viz.png', 'spatioflux_reference_demo_schema.json', 'spatioflux_reference_demo_state.json'] | op all_exist_and_match tolerance 0.0 |


## Study: `reference_demo_x2y2`

**Question.** Different resolution for the spatio-flux reference demo

**Purpose.** Runs the spatio_flux.composites.reference.reference_demo_x2y2 composite (reference demo).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `spatio_flux.composites.reference.reference_demo_x2y2` | 0 | n_bins=[20, 20] |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `spatio_flux.composites.reference.reference_demo_x2y2`**

The SpatioFlux reference model on a finer field lattice with doubled bin counts in x and y: the same Newtonian particles carrying a two-strain internal-dFBA community, with local substrate exchange, diffusing glucose/acetate fields, and sub-mass-splitting division.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_spatio_flux_composites_reference_reference_demo_x2y2 = {
    'bounds'     : [50.0, 50.0],
    'n_bins'     : [20, 20],
    'depth'      : 0.04,
    'n_particles': 1,
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: reference_demo_x2y2 ===
STUDY = 'reference_demo_x2y2'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_spatio_flux_composites_reference_reference_demo_x2y2 if 'ov_spatio_flux_composites_reference_reference_demo_x2y2' in dir() else None  # edits from the panel cell, if any
            spec_spatio_flux_composites_reference_reference_demo_x2y2 = materialize_composite('spatio_flux.composites.reference.reference_demo_x2y2', _ov)
            _comp = Composite({'state': spec_spatio_flux_composites_reference_reference_demo_x2y2['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(spatio_flux.composites.reference.reference_demo_x2y2) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**reference_demo_x2y2.svg**


In [ ]:
# reference_demo_x2y2.svg
show_viz(_render_one('image:visualizations/reference_demo_x2y2.svg', {}, RUNS_DB, STUDY_YAML))

**reference_demo_x2y2_timeseries.png**


In [ ]:
# reference_demo_x2y2_timeseries.png
show_viz(_render_one('image:charts/reference_demo_x2y2_timeseries.png', {}, RUNS_DB, STUDY_YAML))

**reference_demo_x2y2_snapshots.png**


In [ ]:
# reference_demo_x2y2_snapshots.png
show_viz(_render_one('image:charts/reference_demo_x2y2_snapshots.png', {}, RUNS_DB, STUDY_YAML))

**reference_demo_x2y2_mass.png**


In [ ]:
# reference_demo_x2y2_mass.png
show_viz(_render_one('image:charts/reference_demo_x2y2_mass.png', {}, RUNS_DB, STUDY_YAML))

**reference_demo_x2y2_mass_submasses.png**


In [ ]:
# reference_demo_x2y2_mass_submasses.png
show_viz(_render_one('image:charts/reference_demo_x2y2_mass_submasses.png', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| REFERENCE_DEMO_X2Y2-REPRODUCES-REPORT _(regression)_ | kind=artifacts_present expected=['reference_demo_x2y2_timeseries.png', 'reference_demo_x2y2_mass.png', 'reference_demo_x2y2_mass_submasses.png', 'reference_demo_x2y2_video.gif', 'reference_demo_x2y2_snapshots.png', 'reference_demo_x2y2_viz.png', 'reference_demo_x2y2_schema.json', 'reference_demo_x2y2_state.json'] | op all_exist_and_match tolerance 0.0 |
